In [ ]:
pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 54.6 MB/s eta 0:00:00


In [ ]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import time
import urllib3
import fitz
import io

In [ ]:
# Tắt cảnh báo SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class VJSTCrawler:
    def __init__(self, db_name="khoahoc_vn.db"):
        self.db_name = db_name
        self.setup_database()
        self.session = requests.Session()
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36",
        }



    def setup_database(self):
        conn = sqlite3.connect(self.db_name)
        cursor = conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS articles (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT,
                author TEXT,
                abstract TEXT,
                content TEXT,
                url TEXT UNIQUE,
                category TEXT
            )
        ''')
        conn.commit()
        conn.close()




    def extract_text_from_pdf(self, pdf_url):
        # Tải và trích xuất văn bản từ file PDF
        try:
            response = self.session.get(pdf_url, headers=self.headers, verify=False, timeout=25)
            if response.status_code == 200:
                with fitz.open(stream=response.content, filetype="pdf") as doc:
                    # lấy toàn bộ PDF
                    text = "".join([page.get_text() for page in doc])
                    return text.strip()
        except Exception as e:
            print("Lỗi PDF:", e)
        return "N/A"





    def save_to_db(self, data):
        conn = sqlite3.connect(self.db_name)
        cursor = conn.cursor()
        try:
            cursor.execute('''
                INSERT OR IGNORE INTO articles (title, author, abstract, content, url, category)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (data['title'], data['author'], data['abstract'], data['content'], data['url'], data['category']))
            conn.commit()
            return True
        except:
            return False
        finally:
            conn.close()




    def crawl_all(self, total_needed=5):
        cnt = 0
        page = 1
        base_url = "https://b.vjst.vn"
        archive_url = f"{base_url}/index.php/ban_b/issue/archive"

        while True:
            print(f"\n--- Đang quét danh mục trang {page} ---")
            try:
                res = self.session.get(f"{archive_url}/{page}", headers=self.headers, verify=False, timeout=20)
                if res.status_code != 200:
                  break

                soup = BeautifulSoup(res.content, "html.parser")
                issue_links = [a['href'] for a in soup.select("a.title") if 'issue/view' in a['href']]

                if not issue_links:
                    print("Đã hết danh mục số báo.")
                    break

                for issue_url in issue_links:
                    # Truy cập vào từng số tạp chí
                    iss_res = self.session.get(issue_url, headers=self.headers, verify=False, timeout=20)
                    iss_soup = BeautifulSoup(iss_res.content, "html.parser")
                    article_blocks = iss_soup.select(".obj_article_summary")

                    for block in article_blocks:
                        # KIỂM TRA DỪNG: Nếu đã xử lý đủ số lượng bài yêu cầu thì thoát hẳn hàm
                        if cnt >= total_needed:
                            return

                        try:
                            title_tag = block.select_one("h3.title a, .title a")
                            if not title_tag: continue

                            # Tăng biến đếm ngay khi xác nhận có bài báo để xử lý
                            cnt += 1
                            title = title_tag.text.strip()
                            link = title_tag['href']

                            print(f"[{cnt}] Đang xử lý: {title[:60]}...")

                            # Lấy thông tin chi tiết
                            det_res = self.session.get(link, headers=self.headers, verify=False, timeout=20)
                            det_soup = BeautifulSoup(det_res.content, "html.parser")

                            abstract_div = det_soup.select_one("section.item.abstract, .abstract")
                            author_div = det_soup.select_one("section.item.authors, .authors")
                            pdf_link_tag = det_soup.select_one("a.obj_galley_link.pdf")

                            content_text = "N/A"
                            if pdf_link_tag:
                                # Chuyển link viewer sang link download trực tiếp
                                pdf_url = pdf_link_tag['href'].replace("/view/", "/download/")
                                content_text = self.extract_text_from_pdf(pdf_url)

                            data = {
                                "title": title,
                                "author": author_div.get_text().strip() if author_div else "N/A",
                                "abstract": abstract_div.get_text().replace("Tóm tắt", "").strip() if abstract_div else "N/A",
                                "content": content_text,
                                "url": link,
                                "category": "Khoa học Công nghệ"
                            }

                            # Lưu vào database
                            self.save_to_db(data)

                            # Nghỉ 1 giây để tránh spam server
                            time.sleep(1)

                        except Exception as e:
                            print(f"Lỗi bài báo: {e}")
                            continue

                page += 1
            except Exception as e:
                print(f"Lỗi kết nối trang danh mục: {e}")
                break

if __name__ == "__main__":
    crawler = VJSTCrawler()
    crawler.crawl_all(1000)


--- Đang quét danh mục trang 1 ---
[1] Đang xử lý: Ảnh hưởng của chất trợ tương hợp glycidyl methacrylate và kh...
[2] Đang xử lý: Tiêu chuẩn đánh giá chất lượng trong nuôi cấy tế bào gốc và ...
[3] Đang xử lý: Nghiên cứu nhân giống loài Lan Hoàng thảo Thái Bình (Dendrob...
[4] Đang xử lý: Tổng hợp và nghiên cứu tính chất của composite PbO2-SnO2 trê...
[5] Đang xử lý: Chế tạo vật liệu tổ hợp TiO2/HAP bằng phương pháp thủy nhiệt...
[6] Đang xử lý: Cảm biến điện hóa với điện cực làm việc biến tính bằng hạt n...
[7] Đang xử lý: Đánh giá tỷ lệ di căn hạch cổ trong ung thư amidan tại Bệnh ...
[8] Đang xử lý: Đánh giá hiệu quả chương trình đào tạo “Cấp cứu sơ sinh” tại...
[9] Đang xử lý: Thực hành tuân thủ điều trị dự phòng trước phơi nhiễm HIV ở ...
[10] Đang xử lý: Ảnh hưởng của mật độ cấy và lượng phân bón hóa học đến sinh ...
[11] Đang xử lý: Xây dựng hệ thống tái sinh in vitro phục vụ chuyển gen ở giố...
[12] Đang xử lý: Đánh giá tính bền vững của mô hình sinh thái tích hợp trong ...
[